# FGW + Wishart • ConceptNet 100 000 • CUDA + Google Drive

Вход: 100 000 вершин связной выборки ConceptNet, до 25 000 ego-кандидатов. FGW: 16 опорных вершин, alpha 0.5, пять ближайших соседей Уишарта. GPU вычисляет приближённый энтропийный FGW; CPU с автоматически определённым числом ядер выполняет подготовку, VF2, CSR и диагностику.

При небольшом числе канонических типов сравниваются все пары; при большом числе предварительно отбирается до 32 соседей на тип по дешёвому дескриптору. Это приближённый top-5, не полный FGW по всем 25 000 ego-кандидатам. Типов для FGW обычно меньше, чем исходных кандидатов. Результаты и контрольные точки сохраняются в ту же папку Drive, исходный TSV не меняется. Если крупнейшая компонента меньше 100 000, выполнение прекращается, а не подменяет масштаб.

CUDA обязательна: выберите Runtime → Change runtime type → GPU. При недостатке RAM выберите High-RAM.

In [ ]:
FOLDER_ID = "1yE6U3uQvGULUl4g67zey3Hrc8a5NNXIo"
BRANCH = "feature/graphex-exact-fgw-colab"
CONFIG_REL = "configs/wishart_conceptnet_fgw_100k_cuda_colab.yaml"
DATASET_REL = "data/conceptnet_en_100k.tsv"
RUN_NAME = "wishart-fgw-cn100k-gpu-k5-01"
MODE = "NEW"  # NEW или RESUME
RUN_PIPELINE = True
SCRATCH = "/content/semmap-fgw-100k"

## 1. Автоматическая проверка ресурсов и вычислительных ядер

In [ ]:
import os, sys, math, json, shutil, subprocess
from pathlib import Path
assert MODE in {"NEW", "RESUME"}
assert sys.platform == "linux"
scratch = Path(SCRATCH)
scratch.mkdir(parents=True, exist_ok=True)

def effective_cpu_count():
    counts = [os.cpu_count() or 1]
    if hasattr(os, "sched_getaffinity"):
        counts.append(len(os.sched_getaffinity(0)))
    if hasattr(os, "process_cpu_count"):
        counts.append(os.process_cpu_count() or 1)
    quota = Path("/sys/fs/cgroup/cpu.max")
    if quota.is_file():
        tokens = quota.read_text().split()
        if len(tokens) == 2 and tokens[0] != "max":
            counts.append(max(1, math.ceil(int(tokens[0]) / int(tokens[1]))))
    return max(1, min(counts))

CPU_WORKERS = effective_cpu_count()
mem_kib = int(Path("/proc/meminfo").read_text().split("MemAvailable:")[1].split()[0])
ram = mem_kib * 1024
memory_cap = Path("/sys/fs/cgroup/memory.max")
if memory_cap.is_file() and memory_cap.read_text().strip() != "max":
    ram = min(ram, int(memory_cap.read_text().strip()))
free_disk = shutil.disk_usage("/content").free
if ram < 16 * 1024**3:
    raise RuntimeError("Для 100k требуется минимум 16 GiB RAM: переключитесь на High-RAM")
if free_disk < 12 * 1024**3:
    raise RuntimeError("Нужно минимум 12 GiB свободного локального диска /content")
for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[variable] = "1"
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Нет CUDA: выберите Colab GPU runtime и перезапустите ноутбук")
gpu = torch.cuda.get_device_properties(0)
print(json.dumps({"CPU_WORKERS": CPU_WORKERS, "RAM_GiB": round(ram / 1024**3, 1),
    "free_disk_GiB": round(free_disk / 1024**3, 1), "GPU": gpu.name,
    "VRAM_GiB": round(gpu.total_memory / 1024**3, 1)}, indent=2))

## 2. Google Drive: используем ID исходной папки, без предположений о локальном пути

In [ ]:
from google.colab import drive, auth
drive.mount("/content/drive")
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive.readonly"])
api = build("drive", "v3", credentials=credentials, cache_discovery=False)
root_id = api.files().get(fileId="root", fields="id").execute()["id"]
parts, visited, current = [], set(), FOLDER_ID
while current != root_id:
    if current in visited:
        raise RuntimeError("Цикл родителей Drive")
    visited.add(current)
    item = api.files().get(fileId=current, fields="id,name,mimeType,parents", supportsAllDrives=True).execute()
    if item["mimeType"] != "application/vnd.google-apps.folder":
        raise RuntimeError("Неверный ID папки")
    parts.append(item["name"])
    parents = item.get("parents", [])
    if len(parents) != 1:
        raise RuntimeError("Папка должна находиться в MyDrive")
    current = parents[0]
DRIVE_ROOT = Path("/content/drive/MyDrive").joinpath(*reversed(parts))
source = DRIVE_ROOT / DATASET_REL
if not source.is_file():
    raise FileNotFoundError(source)
(DRIVE_ROOT / "runs").mkdir(exist_ok=True)
print("Dataset:", source, "bytes:", source.stat().st_size)
print("Durable results:", DRIVE_ROOT / "runs" / RUN_NAME)

## 3. Git: для RESUME используем точный коммит из input.json; установка перед научными импортами

In [ ]:
REPO = scratch / "repo"
RUN_DIR = DRIVE_ROOT / "runs" / RUN_NAME
if MODE == "RESUME":
    old = RUN_DIR / "input.json"
    if not old.is_file():
        raise FileNotFoundError(old)
    GIT_REF = json.loads(old.read_text(encoding="utf-8"))["code_revision"]
    if not GIT_REF or GIT_REF == "unknown":
        raise RuntimeError("Для RESUME требуется зафиксированный Git-коммит")
else:
    GIT_REF = BRANCH
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "--no-checkout",
                    "https://github.com/SemanticMap/semgraphex.git", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", GIT_REF], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", "FETCH_HEAD"], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
install = [sys.executable, "-m", "pip", "install", "-q"]
if sys.version_info < (3, 13):
    install += ["-c", str(REPO / "requirements/constraints.txt")]
install += ["-e", str(REPO) + "[wishart,notebook]"]
subprocess.run(install, check=True)
subprocess.run([sys.executable, "-c",
    "import numpy, scipy, ot, torch, semmap_haken.wishart_fgw_accel; "
    "print(numpy.__version__, scipy.__version__, ot.__version__, torch.__version__)"], check=True)
print("Code revision:", COMMIT)

## 4. Фиксируем научные параметры и проверяем реальную CUDA-операцию FGW

In [ ]:
import yaml
import numpy as np
from semmap_haken.wishart_fgw_accel import _torch_fgw_batch
CONFIG = REPO / CONFIG_REL
cfg = yaml.safe_load(CONFIG.read_text(encoding="utf-8"))
wish = cfg["wishart"]
assert cfg["dataset"]["max_nodes"] == 100000
assert cfg["dataset"]["component"] == "largest_connected_sample"
assert wish["metric"] == "fgw" and wish["candidate_limit"] == 25000
assert wish["transport_rank"] == 16 and wish["k_neighbors"] == 5
assert wish["fgw_alpha"] == 0.5 and cfg["colab"]["device"] == "cuda"
assert wish["fgw_shortlist"] >= 5
C = np.array([[0, 1], [1, 0]], dtype=float)
F = np.eye(2, dtype=float)
probe = _torch_fgw_batch([(C, F)], [(C, F)], alpha=0.5,
    epsilon=wish["fgw_epsilon"],
    outer_iterations=wish["fgw_outer_iterations"],
    sinkhorn_iterations=wish["fgw_sinkhorn_iterations"], device="cuda")
torch.cuda.synchronize()
assert np.isfinite(probe).all()
print("CUDA FGW smoke OK:", probe.tolist(),
    "GPU peak MiB:", round(torch.cuda.max_memory_allocated() / 1024**2, 1))
print(json.dumps({"max_nodes": 100000, "candidate_limit": 25000,
    "rank": 16, "k": 5, "alpha": 0.5,
    "shortlist": wish["fgw_shortlist"], "cpu_workers": CPU_WORKERS}, indent=2))

## 5. Запуск и восстановление после прерывания

Внутри семантического pipeline CPU обрабатывает граф и словарь; CUDA выполняет FGW-транспорт. Блоки расстояний сохраняются на Drive каждые 2048 пар, состояния иерархии — после каждого уровня. NEW не перезаписывает существующий прогон. RESUME использует неизменные вход, конфиг и код.

In [ ]:
command = [
    sys.executable, "-m", "semmap_haken.wishart_colab_cli",
    "--config", str(CONFIG),
    "--drive-root", str(DRIVE_ROOT),
    "--dataset-drive", DATASET_REL,
    "--scratch-root", str(scratch / "scratch"),
    "--run-name", RUN_NAME,
    "--device", "cuda",
    "--cpu-workers", str(CPU_WORKERS),
    "--blas-threads", "1",
    "--compare-mode", "sha256",
]
if MODE == "RESUME":
    command.append("--resume")
print(" ".join(command))
if RUN_PIPELINE:
    subprocess.run(command, cwd=str(REPO), check=True)
else:
    print("Dry setup only")

## 6. Проверка результатов

In [ ]:
if RUN_DIR.exists():
    print("Run:", RUN_DIR, "COMPLETED:", (RUN_DIR / "COMPLETED").is_file())
    print("Artifacts:", sorted(p.name for p in RUN_DIR.iterdir()))
    for name in ("input_graph_report.json", "COLAB_RUN.json", "FAILED.json"):
        p = RUN_DIR / name
        if p.is_file():
            print(name, p.read_text(encoding="utf-8")[:3000])
    hierarchy_file = RUN_DIR / "hierarchy.json"
    if hierarchy_file.is_file():
        h = json.loads(hierarchy_file.read_text(encoding="utf-8"))
        print(json.dumps({key:h.get(key) for key in ("initial_nodes","final_nodes","levels","metric","stop_reason","elapsed_seconds")}, indent=2))
        assert h["initial_nodes"] == 100000 and h["metric"] == "fgw"
else:
    print("Пока нет результатов. При RUN_PIPELINE=False это ожидаемо.")

## Ограничения

FGW на GPU решается батчевым энтропийным методом, отличающимся от прежнего точного CPU/POT solver; отчёт фиксирует параметр регуляризации и число реально сравниваемых пар. CUDA не используется для VF2, построения CSR и остальных CPU-этапов. До 25 000 ego-кандидатов не означает 25 000 канонических типов, сравниваемых всеми парами. Если GPU или 100k-связная выборка недоступны, ноутбук завершает работу с явной ошибкой.